# Offline triplet extraction

Run this notebook in Google Colab with a free T4 GPU. It uses multilingual NER as a conservative first pass; review relations before importing them into Neo4j. Upload files with the picker or mount Google Drive.

In [ ]:
!pip -q install transformers torch accelerate sentencepiece

In [ ]:
from pathlib import Path
from google.colab import files

INPUT_DIR = Path('/content/input')
INPUT_DIR.mkdir(exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    (INPUT_DIR / name).write_bytes(data)

# Drive alternative:
# from google.colab import drive
# drive.mount('/content/drive')
# INPUT_DIR = Path('/content/drive/MyDrive/graphrag/input')

In [ ]:
from transformers import pipeline
import json
import re

ner = pipeline(
    'token-classification',
    model='Jean-Baptiste/roberta-large-ner-english',
    aggregation_strategy='simple',
    device=0,
)

def clean(value):
    return re.sub(r'\s+', ' ', value).strip()

def extract_triplets(text, source_doc):
    entities = [clean(item['word']) for item in ner(text) if clean(item['word'])]
    return [
        {
            'subject': subject,
            'relation': 'mentioned_with',
            'object': obj,
            'confidence': 0.5,
            'source_doc': source_doc,
            'source_page': None,
        }
        for subject, obj in zip(entities, entities[1:])
        if subject.casefold() != obj.casefold()
    ]

In [ ]:
triplets = []
for path in INPUT_DIR.glob('*.txt'):
    triplets.extend(extract_triplets(path.read_text(encoding='utf-8', errors='ignore'), path.name))

output_path = Path('/content/triplets.json')
output_path.write_text(json.dumps(triplets, ensure_ascii=False, indent=2), encoding='utf-8')
files.download(str(output_path))
print(f'Exported {len(triplets)} triplets')